In [44]:
import sqlite3
import pandas as pd

con = sqlite3.connect(
    r"C:\Documents\ipl-cricket-analytics\data\raw\ipl.db"
)

print("Database connected successfully!")

Database connected successfully!


In [45]:
def q(sql):
    return pd.read_sql_query(sql, con)

print("Query function ready!")

Query function ready!


In [46]:
q("""
SELECT name
FROM sqlite_master
WHERE type = 'table';
""")

,name
0,matches
1,deliveries
2,teams
3,players
4,venues
5,matches_clean


In [47]:
con.executescript("""
DROP VIEW IF EXISTS v_ball;

CREATE VIEW v_ball AS
SELECT
    d.*,
    CASE
        WHEN is_wide_ball = 1 OR is_no_ball = 1 THEN 0
        ELSE 1
    END AS is_legal,
    CASE
        WHEN over_number < 6 THEN 'Powerplay'
        WHEN over_number < 15 THEN 'Middle'
        ELSE 'Death'
    END AS phase
FROM deliveries d;
""")

con.commit()

print("v_ball created successfully!")

v_ball created successfully!


In [48]:
q("""
SELECT *
FROM v_ball
LIMIT 10;
""")

,season_id,match_id,batter,bowler,non_striker,over_number,ball_number,batter_runs,extras,total_runs,...,penalty_runs,wicket_kind,is_super_over,innings,batting_team_id,bowling_team_id,batting_team,bowling_team,is_legal,phase
0,2008,335982,SC Ganguly,P Kumar,BB McCullum,0,0,0,1,1,...,0,None,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,1,Powerplay
1,2008,335982,BB McCullum,P Kumar,SC Ganguly,0,1,0,0,0,...,0,None,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,1,Powerplay
2,2008,335982,BB McCullum,P Kumar,SC Ganguly,0,2,0,1,1,...,0,None,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,Powerplay
3,2008,335982,BB McCullum,P Kumar,SC Ganguly,0,3,0,0,0,...,0,None,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,1,Powerplay
4,2008,335982,BB McCullum,P Kumar,SC Ganguly,0,4,0,0,0,...,0,None,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,1,Powerplay
5,2008,335982,BB McCullum,P Kumar,SC Ganguly,0,5,0,0,0,...,0,None,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,1,Powerplay
6,2008,335982,BB McCullum,P Kumar,SC Ganguly,0,6,0,1,1,...,0,None,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,1,Powerplay
7,2008,335982,BB McCullum,Z Khan,SC Ganguly,1,0,0,0,0,...,0,None,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,1,Powerplay
8,2008,335982,BB McCullum,Z Khan,SC Ganguly,1,1,4,0,4,...,0,None,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,1,Powerplay
9,2008,335982,BB McCullum,Z Khan,SC Ganguly,1,2,4,0,4,...,0,None,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,1,Powerplay


In [49]:
con.executescript("""
DROP VIEW IF EXISTS v_innings;

CREATE VIEW v_innings AS
SELECT
    match_id,
    innings,
    MIN(batting_team) AS batting_team,
    MIN(bowling_team) AS bowling_team,
    SUM(total_runs) AS runs,
    SUM(is_wicket) AS wickets,
    SUM(is_legal) AS legal_balls
FROM v_ball
WHERE innings IN (1, 2)
GROUP BY match_id, innings;
""")

con.commit()

print("v_innings created successfully!")

v_innings created successfully!


In [50]:
q("""
SELECT *
FROM v_innings
LIMIT 10;
""")

,match_id,innings,batting_team,bowling_team,runs,wickets,legal_balls
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,222,3,120
1,335982,2,Royal Challengers Bangalore,Kolkata Knight Riders,82,10,91
2,335983,1,Chennai Super Kings,Punjab Kings,240,5,120
3,335983,2,Punjab Kings,Chennai Super Kings,207,4,120
4,335984,1,Rajasthan Royals,Delhi Capitals,129,8,120
5,335984,2,Delhi Capitals,Rajasthan Royals,132,1,91
6,335985,1,Mumbai Indians,Royal Challengers Bangalore,165,7,120
7,335985,2,Royal Challengers Bangalore,Mumbai Indians,166,5,118
8,335986,1,Sunrisers Hyderabad,Kolkata Knight Riders,110,10,112
9,335986,2,Kolkata Knight Riders,Sunrisers Hyderabad,112,5,114


In [51]:
con.executescript("""
DROP VIEW IF EXISTS v_match_totals;

CREATE VIEW v_match_totals AS
SELECT
    m.*,
    i1.runs AS first_innings_runs,
    i1.batting_team AS bat_first_team,
    CASE
        WHEN m.match_winner = i1.batting_team THEN 0
        ELSE 1
    END AS chase_won
FROM matches m
JOIN v_innings i1
    ON i1.match_id = m.match_id
    AND i1.innings = 1
WHERE m.result = 'win';
""")

con.commit()

print("v_match_totals created successfully!")

v_match_totals created successfully!


In [52]:
q("""
SELECT
    match_id,
    first_innings_runs,
    bat_first_team,
    match_winner,
    chase_won
FROM v_match_totals
LIMIT 10;
""")

,match_id,first_innings_runs,bat_first_team,match_winner,chase_won
0,335982,222,Kolkata Knight Riders,Kolkata Knight Riders,0
1,1082591,207,Sunrisers Hyderabad,Sunrisers Hyderabad,0
2,1082592,184,Mumbai Indians,Rising Pune Supergiant,1
3,1082593,183,Gujarat Lions,Kolkata Knight Riders,1
4,1082594,163,Rising Pune Supergiant,Punjab Kings,1
5,1082595,157,Royal Challengers Bangalore,Royal Challengers Bangalore,0
6,1082596,135,Gujarat Lions,Sunrisers Hyderabad,1
7,1082597,178,Kolkata Knight Riders,Mumbai Indians,1
8,1082598,148,Royal Challengers Bangalore,Punjab Kings,1
9,1082599,205,Delhi Capitals,Delhi Capitals,0


In [53]:
q("""
SELECT *
FROM v_ball
LIMIT 10;
""")

,season_id,match_id,batter,bowler,non_striker,over_number,ball_number,batter_runs,extras,total_runs,...,penalty_runs,wicket_kind,is_super_over,innings,batting_team_id,bowling_team_id,batting_team,bowling_team,is_legal,phase
0,2008,335982,SC Ganguly,P Kumar,BB McCullum,0,0,0,1,1,...,0,None,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,1,Powerplay
1,2008,335982,BB McCullum,P Kumar,SC Ganguly,0,1,0,0,0,...,0,None,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,1,Powerplay
2,2008,335982,BB McCullum,P Kumar,SC Ganguly,0,2,0,1,1,...,0,None,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,Powerplay
3,2008,335982,BB McCullum,P Kumar,SC Ganguly,0,3,0,0,0,...,0,None,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,1,Powerplay
4,2008,335982,BB McCullum,P Kumar,SC Ganguly,0,4,0,0,0,...,0,None,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,1,Powerplay
5,2008,335982,BB McCullum,P Kumar,SC Ganguly,0,5,0,0,0,...,0,None,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,1,Powerplay
6,2008,335982,BB McCullum,P Kumar,SC Ganguly,0,6,0,1,1,...,0,None,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,1,Powerplay
7,2008,335982,BB McCullum,Z Khan,SC Ganguly,1,0,0,0,0,...,0,None,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,1,Powerplay
8,2008,335982,BB McCullum,Z Khan,SC Ganguly,1,1,4,0,4,...,0,None,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,1,Powerplay
9,2008,335982,BB McCullum,Z Khan,SC Ganguly,1,2,4,0,4,...,0,None,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,1,Powerplay


In [54]:
q("""
SELECT *
FROM v_innings
LIMIT 10;
""")

,match_id,innings,batting_team,bowling_team,runs,wickets,legal_balls
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,222,3,120
1,335982,2,Royal Challengers Bangalore,Kolkata Knight Riders,82,10,91
2,335983,1,Chennai Super Kings,Punjab Kings,240,5,120
3,335983,2,Punjab Kings,Chennai Super Kings,207,4,120
4,335984,1,Rajasthan Royals,Delhi Capitals,129,8,120
5,335984,2,Delhi Capitals,Rajasthan Royals,132,1,91
6,335985,1,Mumbai Indians,Royal Challengers Bangalore,165,7,120
7,335985,2,Royal Challengers Bangalore,Mumbai Indians,166,5,118
8,335986,1,Sunrisers Hyderabad,Kolkata Knight Riders,110,10,112
9,335986,2,Kolkata Knight Riders,Sunrisers Hyderabad,112,5,114


In [55]:
q("""
SELECT
    match_id,
    innings,
    batting_team,
    runs
FROM v_innings
ORDER BY runs DESC
LIMIT 10;
""")

,match_id,innings,batting_team,runs
0,1426268,1,Sunrisers Hyderabad,287
1,1473439,1,Sunrisers Hyderabad,286
2,1473505,1,Sunrisers Hyderabad,278
3,1422126,1,Sunrisers Hyderabad,277
4,1422134,1,Kolkata Knight Riders,272
5,1426273,1,Sunrisers Hyderabad,266
6,1529278,2,Punjab Kings,265
7,1529278,1,Delhi Capitals,264
8,598027,1,Royal Challengers Bangalore,263
9,1426268,2,Royal Challengers Bangalore,262


In [56]:
q("""
SELECT
    ROUND(AVG(runs), 2) AS average_score
FROM v_innings;
""")

,average_score
0,161.13


In [57]:
q("""
SELECT
    batting_team,
    SUM(runs) AS total_runs
FROM v_innings
GROUP BY batting_team
ORDER BY total_runs DESC;
""")

,batting_team,total_runs
0,Mumbai Indians,46517
1,Sunrisers Hyderabad,44948
2,Royal Challengers Bangalore,44890
3,Punjab Kings,44352
4,Delhi Capitals,43297
5,Chennai Super Kings,42557
6,Kolkata Knight Riders,42540
7,Rajasthan Royals,39159
8,Gujarat Titans,12196
9,Lucknow Super Giants,11509


In [58]:
q("""
SELECT
    batting_team,
    ROUND(AVG(runs), 2) AS average_score
FROM v_innings
GROUP BY batting_team
ORDER BY average_score DESC;
""")

,batting_team,average_score
0,Gujarat Titans,176.75
1,Lucknow Super Giants,174.38
2,Chennai Super Kings,164.31
3,Punjab Kings,163.66
4,Mumbai Indians,163.22
5,Gujarat Lions,161.87
6,Sunrisers Hyderabad,161.10
7,Royal Challengers Bangalore,160.90
8,Rajasthan Royals,160.49
9,Delhi Capitals,157.44


In [59]:
q("""
SELECT
    batting_team,
    ROUND(AVG(runs), 2) AS average_score
FROM v_innings
GROUP BY batting_team
HAVING AVG(runs) > 160
ORDER BY average_score DESC;
""")

,batting_team,average_score
0,Gujarat Titans,176.75
1,Lucknow Super Giants,174.38
2,Chennai Super Kings,164.31
3,Punjab Kings,163.66
4,Mumbai Indians,163.22
5,Gujarat Lions,161.87
6,Sunrisers Hyderabad,161.10
7,Royal Challengers Bangalore,160.90
8,Rajasthan Royals,160.49


In [60]:
q("""
SELECT
    batting_team,
    MAX(runs) AS highest_score
FROM v_innings
GROUP BY batting_team
ORDER BY highest_score DESC;
""")

,batting_team,highest_score
0,Sunrisers Hyderabad,287
1,Kolkata Knight Riders,272
2,Punjab Kings,265
3,Delhi Capitals,264
4,Royal Challengers Bangalore,263
5,Lucknow Super Giants,257
6,Mumbai Indians,247
7,Chennai Super Kings,246
8,Rajasthan Royals,242
9,Gujarat Titans,233


In [61]:
q("""
SELECT
    batting_team,
    COUNT(*) AS innings_count
FROM v_innings
GROUP BY batting_team
ORDER BY innings_count DESC;
""")

,batting_team,innings_count
0,Mumbai Indians,285
1,Sunrisers Hyderabad,279
2,Royal Challengers Bangalore,279
3,Delhi Capitals,275
4,Kolkata Knight Riders,272
5,Punjab Kings,271
6,Chennai Super Kings,259
7,Rajasthan Royals,244
8,Gujarat Titans,69
9,Lucknow Super Giants,66


In [62]:
q("""
SELECT
    match_id,
    first_innings_runs,
    bat_first_team,
    match_winner
FROM v_match_totals
WHERE first_innings_runs > 200
ORDER BY first_innings_runs DESC;
""")

,match_id,first_innings_runs,bat_first_team,match_winner
0,1426268,287,Sunrisers Hyderabad,Sunrisers Hyderabad
1,1473439,286,Sunrisers Hyderabad,Sunrisers Hyderabad
2,1473505,278,Sunrisers Hyderabad,Sunrisers Hyderabad
3,1422126,277,Sunrisers Hyderabad,Sunrisers Hyderabad
4,1422134,272,Kolkata Knight Riders,Kolkata Knight Riders
...,...,...,...,...
184,1426288,201,Sunrisers Hyderabad,Sunrisers Hyderabad
185,734011,201,Rajasthan Royals,Rajasthan Royals
186,829787,201,Sunrisers Hyderabad,Sunrisers Hyderabad
187,1527674,201,Sunrisers Hyderabad,Royal Challengers Bangalore


In [63]:
q("""
SELECT
    MAX(first_innings_runs) AS highest_first_innings_score
FROM v_match_totals;
""")

,highest_first_innings_score
0,287


In [64]:
q("""
SELECT
    match_winner,
    COUNT(*) AS wins
FROM v_match_totals
GROUP BY match_winner
ORDER BY wins DESC;
""")

,match_winner,wins
0,Mumbai Indians,153
1,Chennai Super Kings,145
2,Royal Challengers Bangalore,138
3,Kolkata Knight Riders,136
4,Sunrisers Hyderabad,128
5,Punjab Kings,125
6,Delhi Capitals,122
7,Rajasthan Royals,120
8,Gujarat Titans,42
9,Lucknow Super Giants,32


In [65]:
q("""
SELECT
    ROUND(
        100.0 * AVG(chase_won),
        2
    ) AS chase_win_percentage
FROM v_match_totals;
""")

,chase_win_percentage
0,54.51


In [66]:
q("""
SELECT
    venue,
    COUNT(*) AS matches,
    ROUND(
        100.0 * AVG(chase_won),
        2
    ) AS chase_win_percentage
FROM v_match_totals
GROUP BY venue
HAVING COUNT(*) >= 30
ORDER BY chase_win_percentage DESC
LIMIT 10;
""")

,venue,matches,chase_win_percentage
0,Sawai Mansingh Stadium,47,68.09
1,Eden Gardens,77,61.04
2,"Rajiv Gandhi International Stadium, Uppal",48,60.42
3,"Wankhede Stadium, Mumbai",57,59.65
4,M Chinnaswamy Stadium,62,58.06
5,"Punjab Cricket Association Stadium, Mohali",35,57.14
6,Feroz Shah Kotla,59,54.24
7,"MA Chidambaram Stadium, Chepauk, Chennai",37,54.05
8,Wankhede Stadium,72,51.39
9,"Narendra Modi Stadium, Ahmedabad",37,51.35


In [67]:
q("""
SELECT
    phase,
    SUM(total_runs) AS total_runs
FROM v_ball
GROUP BY phase
ORDER BY total_runs DESC;
""")

,phase,total_runs
0,Middle,169160
1,Powerplay,116712
2,Death,104056
